In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes datasets \
    sentence-transformers faiss-cpu rouge-score nltk

import os, pickle, numpy as np, re, torch
from collections import Counter

In [ ]:
# ── Cell 2: Mount Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE = 'path/to/your/dataset'  # Update this path to your dataset location

with open(f'{BASE}/corpus.pkl', 'rb') as f: corpus = pickle.load(f)
with open(f'{BASE}/queries.pkl', 'rb') as f: queries = pickle.load(f)
print(f"✅ Loaded: {len(corpus)} passages | {len(queries)} queries")


In [ ]:
# ── Cell 3: Load FAISS ─────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index(f'{BASE}/faiss_index.bin')
print(f"✅ FAISS: {index.ntotal} passages loaded")


In [ ]:
# ── Cell 4: Load LLM (4-bit) ───────────────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ── MODEL SELECTION ──────────────────────────────────────────
USE_LLAMA = True          # Set False if no Llama-3.1 HF access
HF_TOKEN  = "your_huggingface_token"   # Your HuggingFace token

MODEL_NAME = ("meta-llama/Llama-3.1-8B-Instruct" if USE_LLAMA
              else "microsoft/Phi-3-mini-4k-instruct")
token_arg  = HF_TOKEN if USE_LLAMA else None
# ─────────────────────────────────────────────────────────────

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_use_double_quant=True)

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token_arg)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                                  device_map="auto", token=token_arg)
model.eval()
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Model on: {next(model.parameters()).device}")


In [ ]:
# ── Cell 5: Helper Functions ───────────────────────────────────────────────────
def retrieve(question, top_k=5):
    q_emb = embedder.encode([question], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, top_k)
    return [{'passage': corpus[i], 'score': float(s)}
            for i, s in zip(idxs[0], scores[0])]

def _fmt(question, docs=None):
    """Format prompt for Llama-3 or Phi-3."""
    if USE_LLAMA:
        sys = "You are a helpful assistant. Answer questions briefly and accurately."
        if docs:
            ctx = "\n\n".join(f"[Doc {i+1}]: {d['passage']['text']}"
                               for i, d in enumerate(docs))
            user = f"Documents:\n{ctx}\n\nQuestion: {question}\n\nAnswer (1-2 sentences):"
        else:
            user = f"Question: {question}\n\nAnswer (1-2 sentences):"
        return (f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{sys}\n"
                f"<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user}"
                f"<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n")
    else:  # Phi-3
        if docs:
            ctx = "\n\n".join(f"[Doc {i+1}]: {d['passage']['text']}"
                               for i, d in enumerate(docs))
            return f"<|user|>\nDocuments:\n{ctx}\n\nQuestion: {question}\nAnswer briefly:<|end|>\n<|assistant|>\n"
        return f"<|user|>\nQuestion: {question}\nAnswer briefly:<|end|>\n<|assistant|>\n"

def generate(prompt, max_new_tokens=80):
    inp = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=2048).to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=1.0,
                             pad_token_id=tokenizer.eos_token_id)
    gen = out[0][inp['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def norm(s):
    s = re.sub(r'\b(a|an|the)\b', ' ', s.lower())
    return ' '.join(re.sub(r'[^\w\s]', '', s).split())

def em(pred, golds):
    return int(any(norm(g) == norm(pred) for g in golds))

def f1(pred, golds):
    pt = norm(pred).split(); best = 0
    for g in golds:
        gt = norm(g).split()
        c = Counter(pt) & Counter(gt); nc = sum(c.values())
        if nc == 0: continue
        pr = nc/len(pt) if pt else 0; rc = nc/len(gt) if gt else 0
        if pr+rc > 0: best = max(best, 2*pr*rc/(pr+rc))
    return best

def save_ckpt(data, path):
    with open(path, 'wb') as f: pickle.dump(data, f)

def load_ckpt(path):
    if os.path.exists(path):
        with open(path, 'rb') as f: return pickle.load(f)
    return []

In [ ]:
# ── Cell 6: Run Vanilla LLM ────────────────────────────────────────────────────
CKPT = f'{BASE}/checkpoints/vanilla.pkl'
results_van = load_ckpt(CKPT)
start = len(results_van)
print(f"▶ Vanilla: resuming from {start}/{len(queries)}")

for i, q in enumerate(queries[start:], start=start):
    ans = generate(_fmt(q['question']))
    results_van.append({'query_id': q['id'], 'question': q['question'],
                        'answers': q['answers'], 'answer': ans,
                        'em': em(ans, q['answers']), 'f1': f1(ans, q['answers'])})
    if (i+1) % 25 == 0 or i == len(queries)-1:
        save_ckpt(results_van, CKPT)
        print(f"  [{i+1}/{len(queries)}] F1={np.mean([r['f1'] for r in results_van]):.3f} ✅")

save_ckpt(results_van, f'{BASE}/results/p1/vanilla_final.pkl')
print(f"✅ VANILLA DONE | EM={np.mean([r['em'] for r in results_van]):.3f} | F1={np.mean([r['f1'] for r in results_van]):.3f}")



In [ ]:
# ── Cell 7: Run RAG-Clean ──────────────────────────────────────────────────────
CKPT = f'{BASE}/checkpoints/rag_clean.pkl'
results_rag = load_ckpt(CKPT)
start = len(results_rag)
print(f"▶ RAG-Clean: resuming from {start}/{len(queries)}")

for i, q in enumerate(queries[start:], start=start):
    docs = retrieve(q['question'], top_k=5)
    ans  = generate(_fmt(q['question'], docs))
    results_rag.append({'query_id': q['id'], 'question': q['question'],
                        'answers': q['answers'], 'answer': ans,
                        'retrieved_docs': docs,
                        'em': em(ans, q['answers']), 'f1': f1(ans, q['answers'])})
    if (i+1) % 25 == 0 or i == len(queries)-1:
        save_ckpt(results_rag, CKPT)
        print(f"  [{i+1}/{len(queries)}] F1={np.mean([r['f1'] for r in results_rag]):.3f} ✅")

save_ckpt(results_rag, f'{BASE}/results/p1/rag_clean_final.pkl')
van_f1 = np.mean([r['f1'] for r in results_van])
rag_f1 = np.mean([r['f1'] for r in results_rag])
print(f"\n✅ RAG-CLEAN DONE | EM={np.mean([r['em'] for r in results_rag]):.3f} | F1={rag_f1:.3f}")
print(f"   Vanilla F1={van_f1:.3f} | RAG-Clean F1={rag_f1:.3f}")
print(f"   Status: {'✅ RAG > Vanilla (expected)' if rag_f1>van_f1 else '⚠️ Check corpus'}")
print("\n👉 Next: Open Notebook 2 (GPU runtime)")